***Paddy Leaf Disease Classification using CNN***

This notebook presents a complete pipeline for classifying paddy leaf diseases using a custom Convolutional Neural Network (CNN) model. The workflow includes data preprocessing, augmentation, model building, training with real-time monitoring, evaluation through classification metrics and confusion matrix, and generation of a prediction submission file. Trained on the augmented Paddy Doctor dataset, the model achieved a test accuracy of 95.8%, demonstrating strong performance in multi-class disease identification.

Verifying Dataset Structure and Image Counts per Class
----

In [ ]:
# Define the paths to training and test image directories
train_path = '/kaggle/input/paddy-doctor-disease/Augmented and split - 26000 augmented images split into train (80) sets/train/'
test_path  = '/kaggle/input/paddy-doctor-disease/Augmented and split - 26000 augmented images split into train (80) sets/test/'

# Import required libraries
import glob
from pathlib import Path

# Print the number of training images per class
print('train images')
for filepath in glob.glob(train_path + '/*/'):
    files = glob.glob(filepath + '*')  # List all files in the class directory
    print(f"{len(files)} \t {Path(filepath).name}")  # Output the count and class name

# Print the number of test images per class
print('test images')
for filepath in glob.glob(test_path + '/*/'):
    files = glob.glob(filepath + '*')
    print(f"{len(files)} \t {Path(filepath).name}")

Importing Libraries, Defining Hyperparameters, and Dataset Configuration
--------

In [ ]:
# Import core libraries
import numpy as np
import pandas as pd
import pickle
import cv2
import os
from os import listdir

# Import machine learning and deep learning utilities
from sklearn.preprocessing import LabelBinarizer, MultiLabelBinarizer
from sklearn.model_selection import train_test_split

# Import Keras modules for model building
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import BatchNormalization, Conv2D, MaxPooling2D, Activation
from tensorflow.keras.layers import Flatten, Dropout, Dense
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.utils import plot_model
from keras.preprocessing import image
from keras.preprocessing.image import img_to_array
from keras import backend as K

# Import plotting and system utilities
import matplotlib.pyplot as plt
from psutil import virtual_memory

# Set training constants
SEED = 123               # Seed for reproducibility
EPOCHS = 100             # Total number of training epochs
INIT_LR = 1e-3           # Initial learning rate
BS = 32                  # Batch size

# Image configuration
default_image_size = tuple((256, 256))  # Default image resolution
image_size = 0                          # Placeholder (not used yet)
width = 256
height = 256
depth = 3                               # 3 channels (RGB)

# Get number of classes by counting folders in train directory
n_classes = len(glob.glob(train_path + '/*/'))
print(n_classes)

Defining and Compiling the CNN Model Architecture
----

In [ ]:
# Define a function to build and compile a CNN model
def get_model():
    model = Sequential()                     # Initialize a sequential model
    inputShape = (height, width, depth)      # Default input shape (channels_last format)
    chanDim = -1                             # Channel dimension for BatchNormalization

    # Print image data format (for debugging or backend checking)
    print(K.image_data_format())

    # Adjust input shape and channel dimension if using 'channels_first' (e.g., Theano backend)
    if K.image_data_format() == "channels_first":
        inputShape = (depth, height, width)
        chanDim = 1

    # First Conv block
    model.add(Conv2D(32, (3, 3), padding="same", input_shape=inputShape))
    model.add(Activation("relu"))
    model.add(BatchNormalization(axis=chanDim))
    model.add(MaxPooling2D(pool_size=(3, 3)))
    model.add(Dropout(0.25))

    # Second Conv block
    model.add(Conv2D(64, (3, 3), padding="same"))
    model.add(Activation("relu"))
    model.add(BatchNormalization(axis=chanDim))
    model.add(Conv2D(64, (3, 3), padding="same"))
    model.add(Activation("relu"))
    model.add(BatchNormalization(axis=chanDim))
    model.add(MaxPooling2D(pool_size=(2, 2)))
    model.add(Dropout(0.25))

    # Third Conv block
    model.add(Conv2D(128, (3, 3), padding="same"))
    model.add(Activation("relu"))
    model.add(BatchNormalization(axis=chanDim))
    model.add(Conv2D(128, (3, 3), padding="same"))
    model.add(Activation("relu"))
    model.add(BatchNormalization(axis=chanDim))
    model.add(MaxPooling2D(pool_size=(2, 2)))
    model.add(Dropout(0.25))

    # Fully Connected (Dense) layers
    model.add(Flatten())
    model.add(Dense(1024))
    model.add(Activation("relu"))
    model.add(BatchNormalization())
    model.add(Dropout(0.5))

    # Output layer with softmax activation for multi-class classification
    model.add(Dense(n_classes))
    model.add(Activation("softmax"))

    # Compile the model using Adam optimizer and binary crossentropy loss
    opt = Adam(learning_rate=INIT_LR)
    model.compile(loss="binary_crossentropy", optimizer=opt, metrics=["_


Data Augmentation and Data Generators Setup
---------

In [ ]:
# Create an ImageDataGenerator with data augmentation for training and validation
image_datagen = ImageDataGenerator(
    featurewise_center=False,
    samplewise_center=False,
    featurewise_std_normalization=False,
    samplewise_std_normalization=False,
    zca_whitening=False,
    rotation_range=5,               # Randomly rotate images by 5 degrees
    shear_range=0.2,                 # Apply shearing transformations
    zoom_range=0.2,                  # Random zoom inside pictures
    width_shift_range=0.05,          # Horizontal shifts
    height_shift_range=0.05,         # Vertical shifts
    channel_shift_range=0.,          # No channel shift
    fill_mode='nearest',             # Fill in missing pixels after transformations
    horizontal_flip=True,            # Random horizontal flip
    vertical_flip=False,             # No vertical flip
    rescale=1./255,                  # Normalize pixel values to [0, 1]
    validation_split=0.2             # 20% of data used for validation
)

# Create the training data generator
train_generator = image_datagen.flow_from_directory(
    directory=train_path,
    subset='training',               # Use the training subset
    target_size=(256, 256),           # Resize all images to 256x256
    color_mode="rgb",                 # Use RGB color mode
    batch_size=32,                    # Generate batches of 32 images
    class_mode="categorical",         # Use categorical labels
    shuffle=True,                     # Shuffle images every epoch
    seed=SEED                         # Set random seed for reproducibility
)

# Create the validation data generator
valid_generator = image_datagen.flow_from_directory(
    directory=train_path,
    subset='validation',              # Use the validation subset
    target_size=(256, 256),
    color_mode="rgb",
    batch_size=32,
    class_mode="categorical",
    shuffle=True,
    seed=SEED
)

# Create the test data generator (no augmentation, only rescaling)
test_generator = ImageDataGenerator(rescale=1./255).flow_from_directory(
    directory=test_path,
    target_size=(256, 256),
    color_mode="rgb",
    batch_size=1,                     # Single image at a time
    class_mode="categorical",          # Use categorical labels
    shuffle=False,                    # Do not shuffle test data
    seed=SEED
)

# Print class indices to understand label mapping
print(train_generator.class_indices)

# Print the number of samples in train, validation, and test sets
print(train_generator.samples, valid_generator.samples, test_generator.samples)

Weights & Biases (wandb) Setup for Experiment Tracking
---------

In [ ]:
# Import the Weights & Biases (wandb) library for experiment tracking
import wandb

# Log in to wandb using the provided API key
wandb.login(key="")

# Initialize a new wandb run for tracking this experiment
run = wandb.init(
    project="project-ablations",  # Specify the wandb project name
    name="CNN",                   # Set the run name for easier identification
    config={                      # Define the configuration parameters
        "epochs": 100,             # Total number of training epochs
        "base_lr": 0.005,          # Base learning rate
        "weight_decay": 0.01,      # Weight decay (L2 regularization)
        "architecture": "CNN"      # Model architecture being used
    },
    reinit=True                    # Allow reinitializing wandb runs in the same process
)

Callback Setup for Training Monitoring and Checkpointing
----------

In [ ]:
# Install the livelossplot package for real-time loss and accuracy visualization
!pip install livelossplot

# Import PlotLossesCallback for live plotting during training
from livelossplot.inputs.keras import PlotLossesCallback

# Import Keras callbacks for model checkpointing and early stopping
from tensorflow.keras.callbacks import ModelCheckpoint, EarlyStopping

# Initialize the PlotLossesCallback for live loss and metric visualization during training
plot_loss_1 = PlotLossesCallback()

# Define a ModelCheckpoint callback to save the best model weights
tl_checkpoint_1 = ModelCheckpoint(
    filepath='cnn_best.weights.h5',   # File path where best model weights will be saved
    save_weights_only=True,           # Save only model weights, not the full model
    save_best_only=True,              # Save only when the monitored metric improves
    verbose=1                         # Print messages when saving
)

# Define an EarlyStopping callback to stop training early if validation loss does not improve
early_stop = EarlyStopping(
    monitor='val_loss',                # Monitor validation loss
    patience=10,                       # Number of epochs with no improvement after which training will be stopped
    restore_best_weights=True,         # Restore model weights from the epoch with the best value of the monitored quantity
    mode='min'                         # Stop when the monitored quantity has stopped decreasing
)

Model Training with Fine-Tuning and Callbacks
-----------

In [ ]:
# %%time
# Measure the time taken to run this training cell (if running in a notebook)

# Calculate steps per epoch for training and validation
STEP_SIZE_TRAIN = train_generator.n // train_generator.batch_size  # Total training steps per epoch
STEP_SIZE_VALID = valid_generator.n // valid_generator.batch_size  # Total validation steps per epoch

# Retrain the model with fine-tuning
history = model.fit(
    x=train_generator,                 # Training data generator
    steps_per_epoch=STEP_SIZE_TRAIN,    # Number of batches per training epoch
    validation_data=valid_generator,    # Validation data generator
    validation_steps=STEP_SIZE_VALID,   # Number of batches per validation epoch
    callbacks=[tl_checkpoint_1, early_stop, plot_loss_1],  # Use callbacks: checkpoint saving, early stopping, and live plotting
    verbose=1,                          # Verbose output during training
    epochs=EPOCHS                       # Total number of epochs
)

Saving Model Weights and Logging to Weights & Biases
---------

In [ ]:
# Save model weights to a file
model.save_weights("cnn_best.weights.h5")

# Log weights file to W&B
wandb.save("cnn_best.weights.h5")

Plotting Metrics and Logging Results to Weights & Biases
---------

In [ ]:
# %%time
# Measure the time taken to run this cell (if running in a notebook)

# Extract training and validation metrics from the training history
acc = history.history['accuracy']         # Training accuracy over epochs
val_acc = history.history['val_accuracy'] # Validation accuracy over epochs
loss = history.history['loss']             # Training loss over epochs
val_loss = history.history['val_loss']     # Validation loss over epochs
epochs = range(len(acc))                   # Range of epochs

# Log accuracy and loss metrics to Weights & Biases (wandb) per epoch
for epoch in range(len(acc)):
    wandb.log({
        "train_accuracy": acc[epoch],
        "val_accuracy": val_acc[epoch],
        "train_loss": loss[epoch],
        "val_loss": val_loss[epoch],
        "epoch": epoch + 1
    })

# Plot training and validation accuracy
plt.plot(epochs, acc, 'b', label='Training accuracy')
plt.plot(epochs, val_acc, 'r', label='Validation accuracy')
plt.title('Training and Validation Accuracy')
plt.legend()
plt.grid(True)
plt.savefig("accuracy_plot.png")                   # Save accuracy plot to file
wandb.log({"accuracy_plot": wandb.Image("accuracy_plot.png")})  # Log accuracy plot to wandb

# Create a new figure for the loss plot
plt.figure()

# Plot training and validation loss
plt.plot(epochs, loss, 'b', label='Training loss')
plt.plot(epochs, val_loss, 'r', label='Validation loss')
plt.title('Training and Validation Loss')
plt.legend()
plt.grid(True)
plt.savefig("loss_plot.png")                       # Save loss plot to file
wandb.log({"loss_plot": wandb.Image("loss_plot.png")})  # Log loss plot to wandb

# Show plots
plt.show()

Listing Test Images and Creating Test Data Generator
-------

In [ ]:
# Print the number of test images per class
print('test images')
for filepath in glob.glob(test_path + '/*/'):
    files = glob.glob(filepath + '*')  # List all files inside each class directory
    print(f"{len(files)} \t {Path(filepath).name}")  # Print number of files and class name

# Create a test data generator (only rescaling, no augmentation)
test_generator = ImageDataGenerator(rescale=1./255).flow_from_directory(
    directory=test_path,          # Directory containing test images organized by class folders
    target_size=(256, 256),        # Resize all images to 256x256
    color_mode="rgb",              # Load images in RGB mode
    batch_size=1,                  # Load one image at a time
    class_mode="categorical",      # Assign categorical labels
    shuffle=False,                 # Do not shuffle test data to preserve order
    seed=SEED                      # Set random seed for reproducibility
)

Model Prediction on Test Data
-----

In [ ]:
# Calculate the number of test steps based on test data size and batch size
STEP_SIZE_TEST = test_generator.n // test_generator.batch_size

# Reset the test generator to start from the beginning
test_generator.reset()

# Load the best saved model weights before prediction
model.load_weights('cnn_best.weights.h5')

# Run prediction on the test data
pred = model.predict(
    test_generator,            # Generator yielding test images
    steps=STEP_SIZE_TEST,      # Number of prediction steps
    verbose=1                  # Verbose output to show progress
)

# Convert the model's softmax predictions to class indices
pred_classes = np.argmax(pred, axis=1)

Evaluation Metrics: Accuracy and Classification Report
----------

In [ ]:
# Import evaluation metrics from scikit-learn
from sklearn.metrics import accuracy_score
from sklearn.metrics import classification_report

# Retrieve class names and true labels from the test generator
class_names = test_generator.class_indices.keys()  # Get class names from generator
true_classes = test_generator.classes              # Get ground truth labels

# Compute overall accuracy of the model
acc = accuracy_score(true_classes, pred_classes)
print("CNN Model Accuracy : {:.2f}%".format(acc * 100))

# Generate a detailed classification report
cls_report = classification_report(
    true_classes, 
    pred_classes,
    target_names=class_names,  # Use class names for better readability
    digits=5                   # Set decimal precision for scores
)
print(cls_report)

Plotting Confusion Matrix as a Heatmap
-----------

In [ ]:
# Import seaborn and confusion_matrix for plotting
import seaborn as sns
from sklearn.metrics import confusion_matrix

# Get the class names from the test generator
class_names = test_generator.class_indices.keys()

# Define a reusable function to plot confusion matrix heatmaps
def plot_heatmap(y_true, y_pred, class_names, ax, title):
    cm = confusion_matrix(y_true, y_pred)  # Compute the confusion matrix
    sns.heatmap(
        cm, 
        annot=True,                  # Annotate cells with counts
        square=True,                # Force square cells
        xticklabels=class_names,   # Set x-axis tick labels
        yticklabels=class_names,   # Set y-axis tick labels
        fmt='d',                   # Integer formatting for counts
        cmap=plt.cm.Blues,         # Use blue color map
        cbar=False,                # Disable color bar
        ax=ax                      # Plot on the provided axis
    )
    # Format tick labels and axis labels
    ax.set_xticklabels(ax.get_xticklabels(), fontsize=12, rotation=45, ha="right")
    ax.set_yticklabels(ax.get_yticklabels(), fontsize=12)
    ax.set_ylabel('True Label', fontsize=12)
    ax.set_xlabel('Predicted Label', fontsize=12)
    # Optional title can be added if needed
    # ax.set_title(title, fontsize=16)

# Create a single subplot for the confusion matrix
fig, ax = plt.subplots(1, 1, figsize=(6, 6))

# Plot the confusion matrix for the CNN model
plot_heatmap(true_classes, pred_classes, class_names, ax, title="CNN")

# Show the plot
plt.show()

# Print the raw confusion matrix for inspection
cm = confusion_matrix(true_classes, pred_classes)
print(cm)

Final Model Evaluation on Test Set
----

In [ ]:
# Evaluate the model on the test set using the test generator
loss, acc = model.evaluate(
    test_generator,            # Test data generator
    steps=STEP_SIZE_TEST,      # Number of evaluation steps
    verbose=1                  # Display progress during evaluation
)

# Print the final accuracy and loss on the test data
print(acc, loss)

Displaying Class Distribution in the Training Set
-----------

In [ ]:
# Display the number of samples per class in the training set
pd.Series(train_generator.classes).value_counts()

Displaying Class Distribution in the Test Set
----------

In [ ]:
# Display the number of samples per class in the test set
pd.Series(test_generator.classes).value_counts()

Decoding Predictions and Displaying Predicted Class Distribution
-----------

In [ ]:
# Convert predicted probabilities to class indices
predicted_class_indices = np.argmax(pred, axis=1)

# Retrieve the mapping of class labels (e.g., {'class_name': index})
labels = train_generator.class_indices

# Reverse the mapping to get index → class name
labels = {v: k for k, v in labels.items()}

# Map predicted class indices to their corresponding class names
predictions = [labels[k] for k in predicted_class_indices]

# Display the number of predictions per class
pd.Series(predictions).value_counts()

Creating Submission File with Predicted Labels
---------

In [ ]:
# Get the list of test image file paths from the test generator
filenames = test_generator.filenames

# Create a DataFrame with image IDs and their predicted labels
results = pd.DataFrame({
    "image_id": filenames,
    "label": predictions
})

# Extract only the filename (not full path) for each image
results['image_id'] = results['image_id'].apply(lambda x: os.path.basename(x))

# Save the results to a CSV file for submission or evaluation
results.to_csv("submission.csv", index=False)

# Display the first few rows of the results
results.head()

In [ ]:
# Log to W&B
wandb.save("submission.csv")